In [1]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIGURATION
# =========================
TOTAL_SAMPLES = 24000
TRAIN_SIZE = 18000
TEST_SIZE = 3000
FINAL_TEST_SIZE = 3000

# Output folders 
TRAIN_DIR = Path("../../../data/training")
TEST_DIR = Path("../../../data/test")
FINAL_TEST_DIR = Path("../../../data/final test")

for d in (TRAIN_DIR, TEST_DIR, FINAL_TEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]

# =========================
# SAMPLE GENERATION (Normal Load)
# =========================
def gen_normal_load_sample():
    temp = random.uniform(80, 105)         # Safe coolant range
    rpm  = random.uniform(800, 3000)       # Idle to mid-range
    pressure  = 0.6 + (rpm - 700.0) / 9000.0 + random.uniform(-0.03, 0.03)
    vibration = 0.05 + (rpm - 700.0) / 20000.0 + random.uniform(-0.02, 0.02)
    return [temp, pressure, rpm, vibration], "Normal Load"

def build_dataset(n):
    X, y, rows = [], [], []
    seq_counter = 1
    for _ in range(n):
        features, label = gen_normal_load_sample()
        X.append(features)
        y.append(label)
        rows.append({
            "Time": 1,
            "Sequence": seq_counter,
            "Temperature": features[0],
            "Pressure": features[1],
            "RPM": features[2],
            "Vibration": features[3],
            "State": label
        })
        seq_counter += 1
    # Convert to N,1,4 shape
    return np.array(X, dtype=np.float32).reshape(n, 1, 4), np.array(y), pd.DataFrame(rows)

# =========================
# GENERATE & SAVE SPLITS
# =========================
# Training
X_tr, y_tr, df_tr = build_dataset(TRAIN_SIZE)
df_tr.to_csv(TRAIN_DIR / "NormalLoad_training.csv", index=False)
np.save(TRAIN_DIR / "NormalLoad_training_X.npy", X_tr)
np.save(TRAIN_DIR / "NormalLoad_training_y.npy", y_tr)

# Test
X_te, y_te, df_te = build_dataset(TEST_SIZE)
df_te.to_csv(TEST_DIR / "NormalLoad_test.csv", index=False)
np.save(TEST_DIR / "NormalLoad_test_X.npy", X_te)
np.save(TEST_DIR / "NormalLoad_test_y.npy", y_te)

# Final test
X_ft, y_ft, df_ft = build_dataset(FINAL_TEST_SIZE)
df_ft.to_csv(FINAL_TEST_DIR / "NormalLoad_final test.csv", index=False)
np.save(FINAL_TEST_DIR / "NormalLoad_final test_X.npy", X_ft)
np.save(FINAL_TEST_DIR / "NormalLoad_final test_y.npy", y_ft)

print("✅ Normal Load datasets created.")
print("Training:", X_tr.shape, y_tr.shape)
print("Test:    ", X_te.shape, y_te.shape)
print("Final:   ", X_ft.shape, y_ft.shape)


✅ Normal Load datasets created.
Training: (18000, 1, 4) (18000,)
Test:     (3000, 1, 4) (3000,)
Final:    (3000, 1, 4) (3000,)
